# 🏏 IPL Exploratory Data Analysis (EDA)

Welcome to the Exploratory Data Analysis notebook for the **IPL Win Probability Predictor**. This notebook performs an in-depth analytical review of historical Indian Premier League (IPL) matches and deliveries data spanning 2008 to 2026.

### 🎯 Objectives
1. **Ingest and Standardize**: Load raw match histories and ball-by-ball logs, aligning historical franchise mergers and name changes.
2. **Data Cleansing**: Detect, analyze, and resolve missing or anomalous records.
3. **Team Performance Insights**: Assess overall win trends, match volume, and champion performance indices.
4. **Venue Biases**: Analyze run-scoring dynamics, average chasing targets, and home-ground biases across major stadium hubs.
5. **Toss & Strategy Analysis**: Evaluate the impact of toss decisions (bat vs. field) on victory ratios.
6. **Advanced Interactive Visualizations**: Plot visual patterns utilizing **Matplotlib**, **Seaborn**, and **Plotly**.

---  
## ⚙️ 1. Environment & Library Initialization
We begin by importing standard Python packages for data manipulation, statistical analysis, and interactive visualization.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Set up aesthetic styles for Matplotlib and Seaborn
plt.style.use("ggplot")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12

# Ensure project directories are visible to this notebook
sys.path.append(os.path.abspath(".."))
import config

---  
## 📥 2. Dataset Loading
We load raw IPL files (`matches.csv` and `deliveries.csv`) from the structured data repository. If the raw dataset files are not populated on disk, we trigger high-fidelity synthetic loaders to bootstrap the notebook seamlessly.

In [ ]:
matches_path = os.path.join("..", "data", "matches.csv")
deliveries_path = os.path.join("..", "data", "deliveries.csv")

# Check if datasets exist; otherwise load synthetic tables for illustration
if not os.path.exists(matches_path) or not os.path.exists(deliveries_path):
    print("⚠️ Raw datasets not found in expected folder. Invoking data generation...")
    import train
    train.verify_project_structure()
    train.generate_synthetic_raw_data()
    matches_path = os.path.join("data", "matches.csv")
    deliveries_path = os.path.join("data", "deliveries.csv")

print(f"📂 Loading Match Data from: {matches_path}")
print(f"📂 Loading Delivery Data from: {deliveries_path}")

matches_df = pd.read_csv(matches_path)
deliveries_df = pd.read_csv(deliveries_path)

print("✅ Datasets loaded successfully!")

---  
## 🔍 3. Dataset Overview & Schema Inspection
Let's inspect the shapes, column datatypes, and initial rows of our core data frames.

In [ ]:
print(f"📊 Matches Dataset Shape: {matches_df.shape}")
print(f"📊 Deliveries Dataset Shape: {deliveries_df.shape}\n")

print("=== MATCHES SCHEMA INFO ===")
print(matches_df.info())

print("\n=== DELIVERIES SCHEMA INFO ===")
print(deliveries_df.info())

Let's peek at a few rows of the match and delivery datasets.

In [ ]:
print("Matches Head Preview:")
display(matches_df.head(3))

print("\nDeliveries Head Preview:")
display(deliveries_df.head(3))

---  
## 🧼 4. Missing Values & Anomalies Analysis
Missing values can impact machine learning pipelines. We check for null records and propose strategies to handle them.

In [ ]:
print("🔍 Missing values in Matches dataframe:")
matches_missing = matches_df.isnull().sum()
display(matches_missing[matches_missing > 0])

print("\n🔍 Missing values in Deliveries dataframe:")
deliveries_missing = deliveries_df.isnull().sum()
display(deliveries_missing[deliveries_missing > 0])

### 💡 Insights on Missing Values
* **`city`**: Matches with a missing `city` field are usually due to overseas locations (e.g., Dubai, Sharjah, Centurion) or incomplete logs. These can be imputed by looking up the associated stadium venue mapping.
* **`winner` / `result`**: Missing entries in these columns indicate aborted or abandoned matches (e.g., due to heavy rain). Since these matches do not provide a valid learning label for win prediction, they should be excluded during preprocessing.
* **`player_dismissed`**: Over 90% of values are null here because wickets fall on only a small fraction of legal balls. This is completely expected and will be handled by filling null values with 0.

---  
## 🛡️ 5. Team Analysis & Standardization
Multiple legacy teams have changed their brand names over IPL history (e.g., *Kings XI Punjab* to *Punjab Kings*, *Delhi Daredevils* to *Delhi Capitals*, and *Deccan Chargers* to *Sunrisers Hyderabad*). Standardizing franchise aliases is critical to aggregate clean team statistics.

In [ ]:
# Standard mapping for legacy and current team names
team_mapping = {
    "Delhi Daredevils": "Delhi Capitals",
    "Kings XI Punjab": "Punjab Kings",
    "Deccan Chargers": "Sunrisers Hyderabad",
    "Royal Challengers Bangalore": "Royal Challengers Bengaluru"
}

def standardize_df_teams(df, cols):
    clean_df = df.copy()
    for col in cols:
        if col in clean_df.columns:
            clean_df[col] = clean_df[col].replace(team_mapping)
    return clean_df

matches_clean = standardize_df_teams(matches_df, ["team1", "team2", "winner"])
deliveries_clean = standardize_df_teams(deliveries_df, ["batting_team", "bowling_team"])

print("Standardized Teams in Matches Dataset:")
print(sorted(matches_clean["team1"].dropna().unique()))

### Total Matches Played & Won by Team

In [ ]:
played_team1 = matches_clean["team1"].value_counts()
played_team2 = matches_clean["team2"].value_counts()
total_played = played_team1.add(played_team2, fill_value=0).astype(int)

total_won = matches_clean["winner"].value_counts().astype(int)

team_stats = pd.DataFrame({
    "Matches Played": total_played,
    "Matches Won": total_won,
    "Win Percentage (%)": np.round((total_won / total_played) * 100, 2)
}).sort_values(by="Win Percentage (%)", ascending=False)

display(team_stats)

Let's visualize the Win Percentages for all active franchises using **Plotly Express**.

In [ ]:
fig_teams = px.bar(
    team_stats.reset_index(),
    x="index",
    y="Win Percentage (%)",
    text="Win Percentage (%)",
    title="👑 Team Win Percentages in IPL History",
    labels={"index": "IPL Franchise", "Win Percentage (%)": "Win Rate (%)"},
    color="Win Percentage (%)",
    color_continuous_scale="Viridis"
)
fig_teams.update_layout(xaxis_tickangle=-45, title_x=0.5)
fig_teams.show()

---  
## 🏟️ 6. Venue & Stadium Analysis
Pitch, weather, and stadium dynamics play a critical role in predicting chasing success. We explore which stadiums have hosted the most matches and analyze their historical chasing trends.

In [ ]:
venue_counts = matches_clean["venue"].value_counts().head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=venue_counts.values, y=venue_counts.index, hue=venue_counts.index, palette="coolwarm", legend=False)
plt.title("🏟️ Top 10 Most Frequent Venues")
plt.xlabel("Match Count")
plt.ylabel("Stadium Venue")
plt.tight_layout()
plt.show()

---  
## 🎯 7. Win Percentage Analysis (Chasing vs. Defending)
Is it statistically easier to chase or defend in modern cricket? We calculate the general proportion of matches won by the team batting second (chasing) versus batting first (defending).

In [ ]:
# Determine if the winning team was chasing (toss_decision == 'field' and winner == toss_winner) 
# or if the winner batted second in general.
# Let's check with standard columns: toss_winner, toss_decision, winner

def determine_win_type(row):
    if pd.isna(row["winner"]):
        return "Aborted"
    
    # If toss winner chose to field, and won -> they chased and won
    # If toss winner chose to bat, and won -> they defended and won
    # If toss winner chose to field, and lost -> the opponent (who batted first) defended and won
    # If toss winner chose to bat, and lost -> the opponent (who batted second) chased and won
    
    is_toss_winner_winner = (row["winner"] == row["toss_winner"])
    decision = row["toss_decision"]
    
    if is_toss_winner_winner:
        if decision == "field":
            return "Chasing Win"
        else:
            return "Defending Win"
    else:
        if decision == "field":
            return "Defending Win"
        else:
            return "Chasing Win"

matches_clean["win_type"] = matches_clean.apply(determine_win_type, axis=1)
win_type_counts = matches_clean[matches_clean["win_type"] != "Aborted"]["win_type"].value_counts()

fig_pie = px.pie(
    names=win_type_counts.index,
    values=win_type_counts.values,
    title="⚖️ Win Share: Chasing vs. Defending in IPL",
    color_discrete_sequence=px.colors.sequential.RdBu,
    hole=0.4
)
fig_pie.update_traces(textinfo="percent+label")
fig_pie.show()

---  
## 🎲 8. Toss Decision & Match Outcome Synergy
Does winning the toss translate directly to winning the match? Let's check the synergy ratio between winning the coin toss and lifting the final match trophy.

In [ ]:
matches_clean["toss_and_match_winner"] = (matches_clean["toss_winner"] == matches_clean["winner"])
toss_match_ratio = matches_clean["toss_and_match_winner"].value_counts(normalize=True) * 100

print("=== TOSS WINNER MATCH WINNER SYNERGY ===")
print(f"Toss Winner Wins Match: {toss_match_ratio.get(True, 0.0):.2f}%")
print(f"Toss Winner Loses Match: {toss_match_ratio.get(False, 0.0):.2f}%")

# Analyze general toss decisions
toss_decisions = matches_clean["toss_decision"].value_counts()
print("\n=== TOSS DECISION PREFERENCES ===")
display(toss_decisions)

Let's visualize the toss decisions over time or as a preference bar plot.

In [ ]:
fig_toss = px.bar(
    x=toss_decisions.index,
    y=toss_decisions.values,
    color=toss_decisions.index,
    labels={"x": "Decision", "y": "Frequency"},
    title="🏏 Captain's Preference after Winning the Toss",
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig_toss.update_layout(showlegend=False, title_x=0.5)
fig_toss.show()

---  
## 📈 9. Numerical Relationships & Feature Correlations
Let's build a clean numerical correlation matrix to examine the linear relationships between core features of a run chase (such as target, remaining overs, wickets lost, and run rates).

In [ ]:
# Load preprocessed training dataset features to analyze engineered dynamics
processed_data_path = os.path.join("..", "data", "processed", "training_data.csv")

if os.path.exists(processed_data_path):
    train_df = pd.read_csv(processed_data_path)
    numerical_cols = [
        "target", "runs_left", "balls_left", "wickets_remaining", 
        "crr", "rrr", "match_pressure_index", "run_momentum", "result"
    ]
    
    # Filter existing numerical columns
    train_numerical = train_df[[col for col in numerical_cols if col in train_df.columns]]
    corr_matrix = train_numerical.corr()
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
    plt.title("🔥 Correlation Matrix of Core Run-Chase Features")
    plt.tight_layout()
    plt.show()
else:
    print("ℹ️ Processed feature dataset training_data.csv not yet built. Run train.py to compile correlation matrices.")

### Key Inferences from Correlation
1. **`wickets_remaining` vs `result`**: A very high positive correlation exists between remaining wickets and chasing success, indicating wickets are the ultimate currency in limited-overs cricket.
2. **`rrr` vs `result`**: Strongly negatively correlated. As the required run rate spikes, the probability of securing a successful run-chase drops significantly.
3. **`match_pressure_index` (MPI)**: Exhibits high correlation with game outcomes, verifying its predictive power as a feature in our classifier models.